# Notebook 05: Baseline Model (Logistic Regression)
## ML-Based Liquidity and Trading-Risk Intelligence for Indian Equities

**Phase:** Phase 8 Batch 4 (Stages 8.9–8.11)
**Target:** High-Risk Event (`hre_5d_train`, 5-day forward horizon)
**Evaluation Protocol:** 10-Fold Expanding-Window Walk-Forward Validation with 5-Session Purging

### Objectives
1. Evaluate Logistic Regression baseline across 10 expanding-window walk-forward folds.
2. Compare feature scaling methodologies: **StandardScaler** vs. **RobustScaler** on the 20 approved features.
3. Enforce zero temporal data leakage: fit scalers strictly on training folds; purge 5 trading sessions prior to validation.
4. Perform inner chronological threshold calibration to optimize operational F1 score without outer validation leakage.
5. Analyze market capitalization segment performance (Large, Mid, Small Cap) and inspect coefficient signs.


In [1]:
# Imports and Environment Setup
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add repository root to path regardless of working directory
current_dir = Path('.').resolve()
if (current_dir / 'data' / 'processed').exists():
    REPO_ROOT = current_dir
elif (current_dir.parent / 'data' / 'processed').exists():
    REPO_ROOT = current_dir.parent
else:
    REPO_ROOT = current_dir
    for p in [current_dir] + list(current_dir.parents):
        if (p / 'data' / 'processed').exists():
            REPO_ROOT = p
            break

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.modeling.preprocessing import build_logistic_regression_pipeline, get_model_features
from src.modeling.evaluation import ModelEvaluator

print(f'Repository root: {REPO_ROOT}')
print('Environment initialized successfully.')


Repository root: C:\Users\vighn\Downloads\Indian_Equities_Liquidity_Risk_ML-main\Indian_Equities_Liquidity_Risk_ML-main
Environment initialized successfully.


### 1. Approved Baseline Features & Pipeline Architecture

The baseline models utilize the 20 approved non-redundant features identified in Phase 8 Batch 2.
Both pipelines apply `SimpleImputer(strategy='median')` followed by feature scaling and `LogisticRegression(class_weight='balanced', C=1.0, solver='lbfgs')`.


In [2]:
# Inspect the approved feature inventory for Logistic Regression
lr_features = get_model_features('logistic_regression')
print(f'Approved features for Logistic Regression ({len(lr_features)}):\n')
for i, f in enumerate(lr_features, 1):
    print(f'  {i:2d}. {f}')

# Inspect pipeline instances
pipe_standard = build_logistic_regression_pipeline(scaler_type='standard')
pipe_robust = build_logistic_regression_pipeline(scaler_type='robust')
print('\nStandard Pipeline:', pipe_standard)
print('Robust Pipeline:  ', pipe_robust)


Approved features for Logistic Regression (20):

   1. daily_return
   2. return_5d
   3. vol_5d
   4. downside_vol_20d
   5. max_drawdown_20d
   6. rsi_14
   7. log_turnover
   8. turnover_ratio
   9. volume_z20
  10. amihud_ma20
  11. low_volume_streak
  12. daily_range
  13. overnight_gap_abs
  14. gap_z20
  15. cs_spread_ma10
  16. rolling_beta_60d
  17. rolling_corr_60d
  18. nifty_vol_20d
  19. nifty_return_20d
  20. vix_close

Standard Pipeline: Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])
Robust Pipeline:   Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', RobustScaler()),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    ra

### 2. Walk-Forward Validation Results (10 Folds)

We load the out-of-fold validation metrics and out-of-fold predictions generated by the walk-forward evaluation engine.
Metrics are evaluated at both the standard threshold (0.50) and the inner-calibrated optimal F1 threshold.


In [3]:
# Load Batch 4 walk-forward fold metrics
metrics_path = REPO_ROOT / 'data' / 'interim' / 'phase8_batch4' / 'supervised_fold_metrics.csv'
fold_metrics = pd.read_csv(metrics_path)

# Filter for baseline models
lr_metrics = fold_metrics[fold_metrics['model_name'].isin(['logistic_regression_standard', 'logistic_regression_robust'])]
display_cols = [
    'model_name', 'fold', 'val_start', 'val_end', 'val_rows', 'val_prev',
    'pr_auc', 'roc_auc', 'brier_score', 'f1_50', 'calibrated_threshold', 'f1_calib'
]
lr_metrics[display_cols].sort_values(['model_name', 'fold'])


,model_name,fold,val_start,val_end,val_rows,val_prev,pr_auc,roc_auc,brier_score,f1_50,calibrated_threshold,f1_calib
1,logistic_regression_robust,0,2023-05-10,2023-08-02,15780,0.186122,0.591594,0.859451,0.124040,0.571150,0.44,0.568781
6,logistic_regression_robust,1,2023-08-03,2023-10-31,15821,0.187077,0.516080,0.838822,0.137472,0.544504,0.46,0.553191
11,logistic_regression_robust,2,2023-11-01,2024-01-30,15939,0.195079,0.537365,0.841690,0.133991,0.564720,0.44,0.561329
16,logistic_regression_robust,3,2024-01-31,2024-04-30,16157,0.178883,0.495037,0.827193,0.160944,0.519720,0.48,0.515330
21,logistic_regression_robust,4,2024-05-02,2024-07-29,16260,0.170541,0.463455,0.824365,0.137164,0.526086,0.54,0.516945
26,logistic_regression_robust,5,2024-07-30,2024-10-23,16358,0.193606,0.522702,0.826943,0.128071,0.525882,0.46,0.539130
31,logistic_regression_robust,6,2024-10-24,2025-01-20,16435,0.177609,0.514368,0.835098,0.126081,0.533487,0.38,0.531303
36,logistic_regression_robust,7,2025-01-21,2025-04-21,16685,0.179143,0.416527,0.807106,0.176780,0.495109,0.42,0.481292
41,logistic_regression_robust,8,2025-04-22,2025-07-15,16911,0.177577,0.469176,0.801299,0.123825,0.461538,0.58,0.382260
46,logistic_regression_robust,9,2025-07-16,2025-10-10,17011,0.184351,0.463632,0.802327,0.133503,0.484764,0.38,0.507031


### 3. Scaler Comparison: StandardScaler vs RobustScaler

Compare out-of-fold PR-AUC, ROC-AUC, Brier score, and F1 between StandardScaler and RobustScaler across all 10 folds.


In [4]:
# Aggregated summary by scaler
comparison = lr_metrics.groupby('model_name').agg({
    'pr_auc': ['mean', 'std', 'min', 'max'],
    'roc_auc': ['mean', 'std'],
    'brier_score': ['mean'],
    'f1_50': ['mean'],
    'f1_calib': ['mean'],
    'calibrated_threshold': ['mean'],
    'fit_time_sec': ['sum', 'mean']
}).round(4)
comparison


pr_auc                         roc_auc          \
                               mean     std     min     max    mean     std   
model_name                                                                    
logistic_regression_robust    0.499  0.0487  0.4165  0.5916  0.8264  0.0187   
logistic_regression_standard  0.499  0.0487  0.4166  0.5916  0.8265  0.0187   

                             brier_score   f1_50 f1_calib  \
                                    mean    mean     mean   
model_name                                                  
logistic_regression_robust        0.1382  0.5227   0.5157   
logistic_regression_standard      0.1382  0.5227   0.5152   

                             calibrated_threshold fit_time_sec           
                                             mean          sum     mean  
model_name                                                               
logistic_regression_robust                  0.458     199.2910  19.9291  
logistic_regression_standard                0.460      33.6951   3.3695

### 4. Market Capitalization Stratification

Evaluate performance across Large Cap, Mid Cap, and Small Cap segments on pooled out-of-fold predictions.


In [5]:
# Load out-of-fold predictions
preds_path = REPO_ROOT / 'data' / 'interim' / 'phase8_batch4' / 'supervised_predictions.parquet'
preds_df = pd.read_parquet(preds_path)

for model in ['logistic_regression_standard', 'logistic_regression_robust']:
    print(f'=== Market Cap Stratification: {model} ===')
    sub_df = preds_df.rename(columns={'y_true': 'hre_5d_train', f'prob_{model}': 'hre_prob'})
    cap_eval = ModelEvaluator.evaluate_by_cap_segment(
        sub_df, segment_col='cap_segment', target_col='hre_5d_train', prob_col='hre_prob', threshold=0.50
    )
    print(cap_eval.to_string(index=False))
    print()


=== Market Cap Stratification: logistic_regression_standard ===


cap_segment   pr_auc  roc_auc  threshold       f1  precision   recall  balanced_accuracy  brier_score  log_loss    tp    fp     tn    fn  n_obs  n_pos  prevalence
      large 0.613609 0.895729        0.5 0.353165   0.744983 0.231441           0.606855     0.105833  0.345747  2413   826  45759  8013  57011  10426    0.182877
        mid 0.586017 0.890686        0.5 0.624663   0.565572 0.697542           0.787550     0.127532  0.429466 10415  8000  57337  4516  80268  14931    0.186014
      small 0.494373 0.856851        0.5 0.466159   0.309669 0.942389           0.750655     0.242522  0.724103  4253  9481  12014   260  26008   4513    0.173524
        ALL 0.479688 0.821448        0.5 0.523491   0.482678 0.571845           0.717314     0.138271  0.447165 17081 18307 115110 12789 163287  29870    0.182929

=== Market Cap Stratification: logistic_regression_robust ===


cap_segment   pr_auc  roc_auc  threshold       f1  precision   recall  balanced_accuracy  brier_score  log_loss    tp    fp     tn    fn  n_obs  n_pos  prevalence
      large 0.613701 0.895710        0.5 0.353225   0.746518 0.231345           0.606882     0.105844  0.345729  2412   819  45766  8014  57011  10426    0.182877
        mid 0.586049 0.890674        0.5 0.624546   0.565865 0.696805           0.787319     0.127463  0.429258 10404  7982  57355  4527  80268  14931    0.186014
      small 0.494419 0.856859        0.5 0.466338   0.309827 0.942389           0.750817     0.242428  0.723831  4253  9474  12021   260  26008   4513    0.173524
        ALL 0.479676 0.821416        0.5 0.523477   0.482939 0.571443           0.717233     0.138226  0.447013 17069 18275 115142 12801 163287  29870    0.182929



### 5. Directional Coefficient Inspection

Verify the sign and magnitude of logistic regression coefficients to ensure economic validity.
Higher illiquidity (e.g., `amihud_z20`) and higher volatility (`vol_20d`) should carry positive coefficients.


In [6]:
# Fit demonstration model on training split to inspect coefficients
data_path = REPO_ROOT / 'data' / 'processed' / 'features_with_targets.parquet'
df_full = pd.read_parquet(data_path)
train_mask = (df_full['split'] == 'train') & (~df_full['is_warmup']) & (df_full['hre_5d_train'].notna())

X_demo = df_full.loc[train_mask, lr_features]
y_demo = df_full.loc[train_mask, 'hre_5d_train']

pipe_standard.fit(X_demo, y_demo)
coefs = pd.DataFrame({
    'feature': lr_features,
    'coefficient': pipe_standard.named_steps['classifier'].coef_[0]
}).sort_values('coefficient', ascending=False)

print('Logistic Regression Feature Coefficients (StandardScaler):')
print(coefs.to_string(index=False))


Logistic Regression Feature Coefficients (StandardScaler):
          feature  coefficient
   cs_spread_ma10     0.800164
 downside_vol_20d     0.712055
           rsi_14     0.203652
      daily_range     0.199732
 nifty_return_20d     0.186632
   turnover_ratio     0.110571
           vol_5d     0.089550
 rolling_beta_60d     0.063149
low_volume_streak     0.026342
      amihud_ma20     0.010659
overnight_gap_abs     0.006650
          gap_z20    -0.007893
     daily_return    -0.021331
        return_5d    -0.022656
       volume_z20    -0.026573
 max_drawdown_20d    -0.070456
 rolling_corr_60d    -0.175217
    nifty_vol_20d    -0.340906
        vix_close    -0.514176
     log_turnover    -0.832154
